# Module 10 - Milestone: Your First LLM Notebook

Use this notebook after Module 09B pretraining setup and Module 10 trainer tests are passing. The focus is the first real run: prepare a corpus, train a tiny TransformerLM, track validation loss, sample from the result, and run one controlled follow-up experiment.


In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

import g2c
from g2c.artifacts import (
    load_tinystories_text as load_tinystories_corpus_text,
    load_or_encode_tokenized_corpus,
    load_or_encode_tokenized_pair,
)
from g2c.tokenizer import BPETokenizer
from g2c.pretraining import Trainer, empty_training_history, get_lm_batch, split_token_stream
from g2c.transformer import TransformerLM

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
print("MPS available:", torch.backends.mps.is_available())


def format_training_progress(metrics: dict, *, max_steps: int, tokens_per_step: int) -> Markdown:
    step = int(metrics["step"])
    completed_steps = min(max_steps, step + 1)
    width = 28
    filled = min(width, round(width * completed_steps / max_steps))
    bar = "#" * filled + "-" * (width - filled)
    tokens_seen = completed_steps * tokens_per_step
    elapsed_s = float(metrics.get("elapsed_s") or 0.0)
    tokens_s = float(metrics.get("steps_per_s") or 0.0) * tokens_per_step
    val_loss = metrics.get("val_loss")
    val_text = ""
    if val_loss is not None:
        val_text = f" | val loss `{val_loss:.3f}` | val ppl `{math.exp(val_loss):.1f}`"
    return Markdown(
        f"`[{bar}]` `{completed_steps:,}/{max_steps:,}` | "
        f"train loss `{metrics['train_loss']:.3f}`"
        f"{val_text} | "
        f"lr `{metrics['lr']:.2e}` | "
        f"grad norm `{metrics['grad_norm']:.2f}` | "
        f"tokens seen `{tokens_seen:,}` | "
        f"elapsed `{elapsed_s / 60:.1f} min` | "
        f"tokens/s `{tokens_s:,.0f}`"
    )


def format_training_start(name: str, *, max_steps: int, tokens_per_step: int, start_step: int = 0) -> Markdown:
    width = 28
    completed_steps = min(max_steps, start_step)
    filled = min(width, round(width * completed_steps / max_steps))
    bar = "#" * filled + "-" * (width - filled)
    status = "starting first step and validation warmup; the first update can take a minute on MPS"
    if start_step > 0:
        status = f"resuming from checkpoint at step `{start_step:,}`"
    return Markdown(
        f"{name}: `[{bar}]` `{completed_steps:,}/{max_steps:,}` | "
        f"{status} | "
        f"tokens/step `{tokens_per_step:,}` | planned tokens `{max_steps * tokens_per_step:,}`"
    )


## Run Configuration

Change these values before running the training cells. Tokenizers are loaded from Module 04 artifacts instead of trained here; the prep cells report the actual `vocab_size` used by each model.

In [ ]:
trainer_device = "auto"
text_max_chars = 1_000_000
shakespeare_tokenizer_name = "ShakespeareTokenizer"

model_config = {
    "embedding_dim": 128,
    "num_layers": 4,
    "num_heads": 4,
    "max_seq_len": 128,
    "hidden_dim": 512,
}

trainer_config = {
    "batch_size": 32,
    "context_length": 64,
    "max_steps": 2000,
    "max_lr": 3e-4,
    "min_lr": 3e-5,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_iters": 10,
    "log_every": 20,
    "device": trainer_device,
    "optimizer": "adamw",
}
trainer_seed = 0

raw_sample_config = {
    "prompt": "KING:",
    "max_new_tokens": 300,
    "temperature": 0.8,
    "top_k": None,
    "printable_only": False,
    "seed": 1,
}

readable_sample_config = {
    "prompt": "KING:",
    "max_new_tokens": 300,
    "temperature": 0.5,
    "top_k": 20,
    "printable_only": True,
    "seed": 1,
}
tinystories_raw_sample_config = {
    "prompt": "Once upon a time,",
    "max_new_tokens": 300,
    "temperature": 0.8,
    "top_k": None,
    "printable_only": False,
    "seed": 1,
}

tinystories_readable_sample_config = {
    "prompt": "Once upon a time,",
    "max_new_tokens": 300,
    "temperature": 0.5,
    "top_k": 20,
    "printable_only": True,
    "seed": 1,
}

run_tinystories_5m = True
run_tinystories_30m = False
resume_storylm_checkpoints = True
storylm_checkpoint_every = 100
storylm_checkpoint_dir = repo_root / "data" / "checkpoints" / "storylm"
tinystories_train_max_chars = 5_000_000
tinystories_valid_max_chars = 200_000
tinystories_tokenizer_name = "StoryTokenizer"

tinystories_5m_model_config = {
    "embedding_dim": 256,
    "num_layers": 6,
    "num_heads": 8,
    "max_seq_len": 256,
    "hidden_dim": 1024,
}
tinystories_5m_trainer_config = {
    "batch_size": 16,
    "context_length": 256,
    "max_steps": 3000,
    "max_lr": 3e-4,
    "min_lr": 3e-5,
    "warmup_steps": 200,
    "weight_decay": 0.05,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_iters": 10,
    "log_every": 20,
    "device": trainer_device,
    "optimizer": "adamw",
}

tinystories_30m_model_config = {
    "embedding_dim": 512,
    "num_layers": 9,
    "num_heads": 8,
    "max_seq_len": 256,
    "hidden_dim": 2048,
}
tinystories_30m_trainer_config = {
    "batch_size": 8,
    "context_length": 256,
    "max_steps": 5000,
    "max_lr": 3e-4,
    "min_lr": 3e-5,
    "warmup_steps": 200,
    "weight_decay": 0.05,
    "grad_clip": 1.0,
    "eval_every": 100,
    "eval_iters": 5,
    "log_every": 20,
    "device": trainer_device,
    "optimizer": "adamw",
}


## Before the Notebook

Module 03B should already provide `AdamW`, `cosine_with_warmup`, and `clip_grad_norm_`. Module 09B should already provide `get_lm_batch` and `lm_cross_entropy`. For Module 10, implement `Trainer.train_step`. The trainer also depends on your completed `TransformerLM` from Module 09.


In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_training.py -x"
"Then run: .venv/bin/python -m pytest tests/test_pretraining_setup.py -x"
"Then run: .venv/bin/python -m pytest tests/test_pretraining.py -x"
"Question: Which trainer test is the next one failing, and which part of train_step does it point at?"
"Answer: "


In [ ]:
for test_file in ["tests/test_training.py", "tests/test_pretraining_setup.py", "tests/test_pretraining.py"]:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", test_file],
        cwd=repo_root,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0, f"{test_file} is not passing yet."

print("Module 03B, Module 09B, and Module 10 tests passed.")


## Exercise 1 - Prepare TinyShakespeare or Fallback Text

Use the downloaded TinyShakespeare file if setup has it. The fallback keeps the notebook runnable, but TinyShakespeare is the intended starter corpus. Keep this first run small: it proves the trainer works before you spend time on TinyStories.

In [ ]:
def read_text_prefix(path: Path, max_chars: int) -> str:
    with path.open("r", encoding="utf-8") as f:
        return f.read(max_chars)


def load_pretraining_text(max_chars: int = 200_000) -> str:
    path = repo_root / "data" / "tinyshakespeare.txt"
    if path.exists():
        return read_text_prefix(path, max_chars)

    base = """
    FIRST STUDENT:
    the model predicts the next token from the context.
    SECOND STUDENT:
    the context helps the model choose a better next token.
    FIRST STUDENT:
    gradients update embeddings and transformer blocks.
    """
    return ("\n".join(line.strip() for line in base.strip().splitlines()) + "\n") * 1000


def make_encode_progress(label: str):
    start = time.perf_counter()
    handle = None

    def show(message: Markdown) -> None:
        nonlocal handle
        if handle is None:
            handle = display(message, display_id=True)
        else:
            handle.update(message)

    def update(info: dict) -> None:
        phase = info.get("phase")
        elapsed = time.perf_counter() - start
        if phase == "encode_count_start":
            show(Markdown(f"{label}: preparing chunked encode..."))
        if phase == "encode_count_chunk":
            show(
                Markdown(
                    f"{label}: counting tokens | chunk `{info['chunk_index']:,}` "
                    f"| chars `{info['chars_seen']:,}/{info['chars']:,}` "
                    f"| tokens `{info['tokens_seen']:,}` "
                    f"| elapsed `{elapsed:.1f}s`"
                )
            )
        elif phase == "encode_write_chunk":
            show(
                Markdown(
                    f"{label}: writing tensor | chunk `{info['chunk_index']:,}/{info['chunks']:,}` "
                    f"| tokens `{info['tokens_written']:,}/{info['token_count']:,}` "
                    f"| elapsed `{elapsed:.1f}s`"
                )
            )
        elif phase == "encode_done":
            show(
                Markdown(
                    f"{label}: encoded `{info['token_count']:,}` tokens "
                    f"from `{info['chunks']:,}` chunks | elapsed `{elapsed:.1f}s`"
                )
            )

    return update


text = load_pretraining_text(max_chars=text_max_chars)
tokenizer, encoded_ids = load_or_encode_tokenized_corpus(
    text,
    shakespeare_tokenizer_name,
    label="tinyshakespeare",
    repo_root=repo_root,
    progress_callback=make_encode_progress(f"tinyshakespeare: {shakespeare_tokenizer_name}"),
)
all_ids = encoded_ids
vocab_size = len(tokenizer.vocab)
train_ids, val_ids = split_token_stream(all_ids, train_fraction=0.9)

print("characters:", len(text))
print("tokens:", len(all_ids))
print("vocab size:", vocab_size)
print("uniform-loss baseline:", math.log(vocab_size))
print("train tokens:", len(train_ids))
print("val tokens:", len(val_ids))
assert len(val_ids) > 128

In [ ]:
"Question: Why can a larger BPE vocabulary reduce token count but make the output logits tensor wider?"
"Answer: "
"Question: Why should validation text come from held-out token positions rather than the same windows used for training?"
"Answer: "

## Exercise 2 - Train a Tiny TransformerLM

This is the first real pretraining loop now that Module 09B owns the batch and loss setup. The default below is a small payoff run rather than a smoke test: it should take longer, but the sample should move past pure token soup. If you only want to verify wiring, reduce `max_steps`, `embedding_dim`, and `num_layers` temporarily.

In [ ]:
torch.manual_seed(0)
model = TransformerLM(vocab_size=vocab_size, **model_config)

trainer = Trainer(
    model,
    **trainer_config,
    generator=torch.Generator().manual_seed(trainer_seed),
)

tokens_per_step = trainer.batch_size * trainer.context_length
total_training_tokens = trainer.max_steps * tokens_per_step
print("training device:", trainer.device)
progress = display(
    format_training_start(
        "TinyShakespeare",
        max_steps=trainer.max_steps,
        tokens_per_step=tokens_per_step,
    ),
    display_id=True,
)


def log_training_progress(metrics: dict) -> None:
    progress.update(
        format_training_progress(
            metrics,
            max_steps=trainer.max_steps,
            tokens_per_step=tokens_per_step,
        )
    )


history = trainer.train(train_ids, val_ids, on_log=log_training_progress)
print("final train loss:", history["train_loss"][-1])
print("final val loss:", history["val_loss"][-1])
print("final val perplexity:", math.exp(history["val_loss"][-1]))
print("training tokens seen:", total_training_tokens)

In [ ]:
def plot_training_history(history: dict[str, list], *, baseline_loss: float | None = None) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history["step"], history["train_loss"], label="train")
    if history["val_loss"]:
        axes[0].plot(history["val_step"], history["val_loss"], marker="o", label="val")
    if baseline_loss is not None:
        axes[0].axhline(baseline_loss, color="gray", linestyle="--", label="log(V)")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross entropy")
    axes[0].legend()

    axes[1].plot(history["step"], history["lr"])
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("learning rate")

    axes[2].plot(history["step"], history["grad_norm"])
    axes[2].set_xlabel("step")
    axes[2].set_ylabel("pre-clip grad norm")

    fig.tight_layout()
    plt.show()


plot_training_history(history, baseline_loss=math.log(vocab_size))

In [ ]:
checkpoint_path = repo_root / "data" / "module10-first-llm.pt"
checkpoint_path.parent.mkdir(exist_ok=True)
torch.save(
    {
        "vocab_size": vocab_size,
        "tokenizer_name": shakespeare_tokenizer_name,
        "training_tokens_seen": total_training_tokens,
        "model_config": {
            "embedding_dim": model.embedding_dim,
            "num_layers": model.num_layers,
            "num_heads": model.num_heads,
            "max_seq_len": model.max_seq_len,
            "hidden_dim": model.blocks[0].ffn.hidden_dim if model.blocks else None,
        },
        "params": [p.detach().cpu() for p in model.parameters()],
        "history": history,
    },
    checkpoint_path,
)
print("saved checkpoint:", checkpoint_path)


In [ ]:
"Question: Does validation loss track training loss, or is one moving much faster?"
"Answer: "
"Question: Are gradient norms largest early in training? What does clipping seem to be doing?"
"Answer: "

## Exercise 3 - Sample During or After Training

Module 11 will build the real generation utilities. For now, use a minimal local sampler so you can qualitatively inspect what pretraining learned.

Byte-level BPE can sample raw byte tokens before the model is well trained. The raw escaped sample shows that behavior directly; the readable sample uses top-k sampling and blocks control/invalid-byte tokens so the output is easier to inspect.

In [ ]:
def token_is_readable(tokenizer: BPETokenizer, token_id: int) -> bool:
    piece = tokenizer.decode([token_id])
    if not piece or "�" in piece:
        return False
    return all(ch in "\n\t" or ch.isprintable() for ch in piece)


def readable_token_mask(tokenizer: BPETokenizer) -> torch.Tensor:
    return torch.tensor(
        [token_is_readable(tokenizer, token_id) for token_id in range(len(tokenizer.vocab))],
        dtype=torch.bool,
    )


def apply_top_k(logits: torch.Tensor, top_k: int | None) -> torch.Tensor:
    if top_k is None:
        return logits
    k = min(top_k, logits.numel())
    cutoff = torch.topk(logits, k).values[-1]
    return logits.masked_fill(logits < cutoff, float("-inf"))


@torch.no_grad()
def sample_text(
    model: TransformerLM,
    tokenizer: BPETokenizer,
    prompt: str,
    *,
    max_new_tokens: int = 200,
    temperature: float = 0.5,
    top_k: int | None = 20,
    printable_only: bool = True,
    seed: int = 0,
) -> str:
    device = next(iter(model.parameters())).device
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device)
    generator = torch.Generator().manual_seed(seed)
    printable_mask = readable_token_mask(tokenizer) if printable_only else None

    for _ in range(max_new_tokens):
        ctx = ids[-model.max_seq_len :].unsqueeze(0)
        logits = model(ctx)[0, -1].clone()

        if printable_mask is not None:
            mask = printable_mask.to(device=logits.device)
            logits = logits.masked_fill(~mask, float("-inf"))

        if temperature == 0.0:
            next_id = logits.argmax().reshape(1)
        else:
            logits = apply_top_k(logits / temperature, top_k)
            probs = torch.softmax(logits, dim=-1).detach().cpu()
            next_id = torch.multinomial(probs, 1, generator=generator).to(device)
        ids = torch.cat([ids, next_id.to(device=ids.device)])

    return tokenizer.decode(ids.detach().cpu().tolist())


def escaped_text(text: str) -> str:
    return text.encode("unicode_escape", errors="backslashreplace").decode("ascii")


def show_sample_outputs(
    name: str,
    model: TransformerLM,
    tokenizer: BPETokenizer,
    raw_config: dict,
    readable_config: dict,
) -> None:
    raw_sample = sample_text(model, tokenizer, **raw_config)
    readable_sample = sample_text(model, tokenizer, **readable_config)

    print(f"{name} raw sample, escaped so control bytes do not wreck the notebook output")
    print("-" * 72)
    print(escaped_text(raw_sample))
    print(f"\n{name} readable top-k sample")
    print("-" * 72)
    print(readable_sample)


show_sample_outputs("TinyShakespeare", model, tokenizer, raw_sample_config, readable_sample_config)

In [ ]:
"Question: How different is the raw escaped sample from the readable filtered sample?"
"Answer: "

"Question: What improved first: punctuation and line shape, local words, or global meaning?"
"Answer: "

"Question: If the readable sample is still incoherent, is that more likely a sampling issue or a training-budget/model-size issue?"
"Answer: "

## Exercise 4 - TinyStories Scale-Up

TinyShakespeare proves the loop works, but it is too small for larger models. If you have downloaded TinyStories with `./datasets.sh tinystories` and generated `StoryTokenizer` in Module 04, run the ~5M-parameter experiment first, inspect the curves, and compare raw/readable samples. After that curve looks healthy, flip `run_tinystories_30m = True` in the config cell and repeat the same train/curves/samples loop for the ~30M-parameter experiment. StoryLM runs save rolling checkpoints every `storylm_checkpoint_every` steps, so you can interrupt, sample the current model, then re-run the training cell to continue. If TinyStories or `StoryTokenizer` is missing, these cells skip cleanly.

In [ ]:
def load_tinystories_text(
    *,
    train_max_chars: int,
    valid_max_chars: int,
) -> tuple[str, str] | None:
    text_pair = load_tinystories_corpus_text(
        train_max_chars=train_max_chars,
        valid_max_chars=valid_max_chars,
        repo_root=repo_root,
    )
    if text_pair is None:
        print("Skipping TinyStories scale-up. Run: ./datasets.sh tinystories")
        return None
    return text_pair


def count_parameters(model: TransformerLM) -> int:
    return sum(p.numel() for p in model.parameters())


def train_scaleup_run(
    name: str,
    *,
    checkpoint_name: str,
    tokenizer: BPETokenizer,
    train_ids: torch.Tensor,
    val_ids: torch.Tensor,
    model_config: dict,
    trainer_config: dict,
    seed: int,
    resume_checkpoint: bool = True,
    checkpoint_every: int = 100,
) -> tuple[TransformerLM, dict[str, list]]:
    torch.manual_seed(seed)
    scale_model = TransformerLM(vocab_size=len(tokenizer.vocab), **model_config)
    scale_trainer = Trainer(
        scale_model,
        **trainer_config,
        generator=torch.Generator().manual_seed(seed),
    )
    tokens_per_step = scale_trainer.batch_size * scale_trainer.context_length
    total_training_tokens = scale_trainer.max_steps * tokens_per_step
    checkpoint_path = storylm_checkpoint_dir / f"{checkpoint_name}.pt"
    checkpoint_extra = {
        "name": name,
        "checkpoint_name": checkpoint_name,
        "tokenizer_name": tinystories_tokenizer_name,
        "vocab_size": len(tokenizer.vocab),
        "model_config": model_config,
        "trainer_config": trainer_config,
        "seed": seed,
    }
    scale_history = empty_training_history()
    if resume_checkpoint and checkpoint_path.exists():
        try:
            checkpoint = scale_trainer.load_checkpoint(checkpoint_path)
        except ValueError as exc:
            raise ValueError(
                f"Existing checkpoint does not match the current {name} config. "
                f"Move or delete {checkpoint_path}, or set resume_storylm_checkpoints = False."
            ) from exc
        scale_history = checkpoint.get("history", empty_training_history())
        print(f"resumed checkpoint at step {scale_trainer.step:,}: {checkpoint_path}")
    else:
        print(f"checkpoint path: {checkpoint_path}")
    print(f"{name} params: {count_parameters(scale_model):,}")
    print("training device:", scale_trainer.device)
    progress = display(
        format_training_start(
            name,
            max_steps=scale_trainer.max_steps,
            tokens_per_step=tokens_per_step,
            start_step=scale_trainer.step,
        ),
        display_id=True,
    )

    def log_progress(metrics: dict) -> None:
        progress.update(
            format_training_progress(
                metrics,
                max_steps=scale_trainer.max_steps,
                tokens_per_step=tokens_per_step,
            )
        )

    try:
        scale_history = scale_trainer.train(
            train_ids,
            val_ids,
            history=scale_history,
            on_log=log_progress,
            checkpoint_path=checkpoint_path,
            checkpoint_every=checkpoint_every,
            checkpoint_extra=checkpoint_extra,
        )
    except KeyboardInterrupt:
        print(f"interrupted at step {scale_trainer.step:,}; saved checkpoint: {checkpoint_path}")

    scale_trainer.save_checkpoint(
        checkpoint_path,
        history=scale_history,
        extra=checkpoint_extra,
    )
    if scale_history["train_loss"]:
        print("latest train loss:", scale_history["train_loss"][-1])
    if scale_history["val_loss"]:
        print("latest val loss:", scale_history["val_loss"][-1])
        print("latest val perplexity:", math.exp(scale_history["val_loss"][-1]))
    print("current step:", scale_trainer.step)
    print("training tokens seen:", scale_trainer.step * tokens_per_step)
    print("planned training tokens:", total_training_tokens)
    print("saved checkpoint:", checkpoint_path)
    return scale_model, scale_history


### TinyStories Tokenizer Prep

Run this once before the TinyStories 5M or 30M training cells. It loads the corpus slice, loads the Module 04 `StoryTokenizer` artifact, encodes train/validation IDs, and leaves the tensors in memory for both model sizes.

In [ ]:
tinystories_text = load_tinystories_text(
    train_max_chars=tinystories_train_max_chars,
    valid_max_chars=tinystories_valid_max_chars,
)
tinystories_ready = tinystories_text is not None

if tinystories_ready:
    try:
        tinystories_train_text, tinystories_valid_text = tinystories_text
        tinystories_tokenizer, tinystories_train_ids, tinystories_val_ids = load_or_encode_tokenized_pair(
            tinystories_train_text,
            tinystories_valid_text,
            tinystories_tokenizer_name,
            label="tinystories",
            repo_root=repo_root,
            train_progress_callback=make_encode_progress(f"tinystories train: {tinystories_tokenizer_name}"),
            val_progress_callback=make_encode_progress(f"tinystories val: {tinystories_tokenizer_name}"),
        )
        print("TinyStories train characters:", len(tinystories_train_text))
        print("TinyStories valid characters:", len(tinystories_valid_text))
        print("TinyStories train tokens:", len(tinystories_train_ids))
        print("TinyStories valid tokens:", len(tinystories_val_ids))
        print("TinyStories vocab size:", len(tinystories_tokenizer.vocab))
    except FileNotFoundError as exc:
        print(exc)
        print("Skipping TinyStories scale-up until the StoryTokenizer artifact exists.")
        tinystories_ready = False
        tinystories_tokenizer = None
        tinystories_train_ids = None
        tinystories_val_ids = None
else:
    tinystories_tokenizer = None
    tinystories_train_ids = None
    tinystories_val_ids = None


### TinyStories 5M Training

This cell uses the already-prepared TinyStories token IDs. Re-running it continues from `storylm-5m.pt` when `resume_storylm_checkpoints` is enabled; set that flag to `False` if you want a fresh run.

In [ ]:
tinystories_5m_model = None
tinystories_5m_history = None

if tinystories_ready and run_tinystories_5m:
    assert tinystories_tokenizer is not None
    assert tinystories_train_ids is not None
    assert tinystories_val_ids is not None
    tinystories_5m_model, tinystories_5m_history = train_scaleup_run(
        "TinyStories 5M",
        checkpoint_name="storylm-5m",
        tokenizer=tinystories_tokenizer,
        train_ids=tinystories_train_ids,
        val_ids=tinystories_val_ids,
        model_config=tinystories_5m_model_config,
        trainer_config=tinystories_5m_trainer_config,
        seed=1,
        resume_checkpoint=resume_storylm_checkpoints,
        checkpoint_every=storylm_checkpoint_every,
    )
elif tinystories_ready:
    print("TinyStories 5M run disabled. Set run_tinystories_5m = True in the config cell.")


### TinyStories 5M Curves and Samples

Run these immediately after the 5M training cell. The curve tells you whether training is healthy; the samples tell you what that loss feels like as text.

In [ ]:
if tinystories_5m_history is not None:
    assert tinystories_tokenizer is not None
    plot_training_history(tinystories_5m_history, baseline_loss=math.log(len(tinystories_tokenizer.vocab)))
elif tinystories_ready:
    print("Run the TinyStories 5M training cell before plotting curves.")


In [ ]:
if tinystories_5m_model is not None:
    assert tinystories_tokenizer is not None
    show_sample_outputs(
        "TinyStories 5M",
        tinystories_5m_model,
        tinystories_tokenizer,
        tinystories_raw_sample_config,
        tinystories_readable_sample_config,
    )
elif tinystories_ready:
    print("Run the TinyStories 5M training cell before sampling.")


### TinyStories 30M Training

Enable `run_tinystories_30m` in the config cell before running this. It reuses the same TinyStories tokenizer and token IDs from the prep cell, and continues from `storylm-30m.pt` when `resume_storylm_checkpoints` is enabled.

In [ ]:
tinystories_30m_model = None
tinystories_30m_history = None

if tinystories_ready and run_tinystories_30m:
    assert tinystories_tokenizer is not None
    assert tinystories_train_ids is not None
    assert tinystories_val_ids is not None
    tinystories_30m_model, tinystories_30m_history = train_scaleup_run(
        "TinyStories 30M",
        checkpoint_name="storylm-30m",
        tokenizer=tinystories_tokenizer,
        train_ids=tinystories_train_ids,
        val_ids=tinystories_val_ids,
        model_config=tinystories_30m_model_config,
        trainer_config=tinystories_30m_trainer_config,
        seed=2,
        resume_checkpoint=resume_storylm_checkpoints,
        checkpoint_every=storylm_checkpoint_every,
    )
elif tinystories_ready:
    print("TinyStories 30M run disabled. Set run_tinystories_30m = True in the config cell after the 5M run looks healthy.")


### TinyStories 30M Curves and Samples

Only run this after enabling `run_tinystories_30m`. It uses the same tokenizer, prompt, curve plot, and raw/readable sample comparison as the 5M run, so the qualitative difference is easier to judge.

In [ ]:
if tinystories_30m_history is not None:
    assert tinystories_tokenizer is not None
    plot_training_history(tinystories_30m_history, baseline_loss=math.log(len(tinystories_tokenizer.vocab)))
elif tinystories_ready:
    print("Run the TinyStories 30M training cell before plotting curves.")


In [ ]:
if tinystories_30m_model is not None:
    assert tinystories_tokenizer is not None
    show_sample_outputs(
        "TinyStories 30M",
        tinystories_30m_model,
        tinystories_tokenizer,
        tinystories_raw_sample_config,
        tinystories_readable_sample_config,
    )
elif tinystories_ready:
    print("Run the TinyStories 30M training cell before sampling.")


In [ ]:
"Question: How does the TinyStories 5M validation curve compare with the TinyShakespeare curve?"
"Answer: "

"Question: If you ran the 30M model, did validation keep improving longer or overfit sooner?"
"Answer: "

"Question: What changed more visibly in samples: spelling, sentence shape, or story coherence?"
"Answer: "